In [42]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# Import the Blind Insight client
import sys
sys.path.append('.')
from blind_insight_client import BlindInsightClient, load_iris_from_blind

# Batch means (integer-scaled) over all rows, in batches using transaction_id
from blind_insight_client import BlindInsightClient

import numpy as np
import warnings

# Suppress SSL warnings for local development
warnings.filterwarnings('ignore', message='Unverified HTTPS request')

# Import the Blind Insight client
import sys
sys.path.append('.')
from blind_insight_client import BlindInsightClient


In [43]:
# Encrypted avg of sepal-width per class (integer-scaled data)
from blind_insight_client import BlindInsightClient

# Configuration - Plaintext ML example (illustrative)
# NOTE: This decrypts data; for encrypted-only workflows see the section below.
ORGANIZATION = "demo"
TRAINING_DATASET_SLUG = "fraud-analysis-training"
API_URL = "https://proxy.local.blindinsight.io/"

USERNAME = "data_owner@localhost"  # Your Blind Insight account email
PASSWORD = "blindinsight"  # Your Blind Insight account password

SCHEMA_SLUG = "fraud-analysis-schema"

REAL_DATASET_SLUG = "fraud-analysis"
TRAINING_DATASET_SLUG = "fraud-analysis-training"

REAL_DATA_SCHEMA_ID = "i9yp3TutxNsCp3fgNWE4p2"
TRAINING_SCHEMA_ID = "iEbXomadeyp3JcDfZMFLYp"

In [44]:
def agg_value(resp):
    recs = resp.get("records", [])
    if not recs:
        raise ValueError(f"Unexpected aggregation response: {resp}")
    rec0 = recs[0]
    if "data" in rec0 and isinstance(rec0["data"], dict) and "value" in rec0["data"]:
        v = rec0["data"].get("value")
        return float(v) if v is not None else 0.0
    if "value" in rec0:
        v = rec0.get("value")
        return float(v) if v is not None else 0.0
    raise ValueError(f"Unexpected aggregation response shape: {resp}")

In [45]:
# Batch means (integer-scaled) over all rows, in batches using transaction_id
from blind_insight_client import BlindInsightClient

import numpy as np
import warnings

# Suppress SSL warnings for local development
warnings.filterwarnings('ignore', message='Unverified HTTPS request')

# Import the Blind Insight client
import sys
sys.path.append('.')
from blind_insight_client import BlindInsightClient
import pandas as pd

client = BlindInsightClient(api_url=API_URL, username=USERNAME, password=PASSWORD, verify_ssl=False)
features = [
    "amount",
    "transaction_type_ATM",
    "transaction_type_Online",
    "transaction_type_POS",
    "transaction_type_QR",
    "merchant_category_Clothing",
    "merchant_category_Electronics",
    "merchant_category_Food",
    "merchant_category_Grocery",
    "merchant_category_Travel",
    "country_DE",
    "country_FR",
    "country_NG",
    "country_TR",
    "country_UK",
    "country_US",
    "hour",
    "is_fraud",
]
schema = {"type": "object", "properties": {"hour": {"type": "integer", "maximum": 23, "minimum": 0}, "amount": {"type": "integer", "maximum": 12000, "minimum": 0}, "user_id": {"type": "integer", "maximum": 1000, "minimum": 0}, "is_fraud": {"type": "integer", "maximum": 1, "minimum": 0}, "country_DE": {"type": "integer", "maximum": 1, "minimum": 0}, "country_FR": {"type": "integer", "maximum": 1, "minimum": 0}, "country_NG": {"type": "integer", "maximum": 1, "minimum": 0}, "country_TR": {"type": "integer", "maximum": 1, "minimum": 0}, "country_UK": {"type": "integer", "maximum": 1, "minimum": 0}, "country_US": {"type": "integer", "maximum": 1, "minimum": 0}, "dataset_order": {"type": "integer", "maximum": 500, "minimum": 0}, "ip_risk_score": {"type": "integer", "maximum": 100, "minimum": 0}, "transaction_id": {"type": "integer", "maximum": 10000, "minimum": 0}, "device_risk_score": {"type": "integer", "maximum": 100, "minimum": 0}, "transaction_type_QR": {"type": "integer", "maximum": 1, "minimum": 0}, "transaction_type_ATM": {"type": "integer", "maximum": 1, "minimum": 0}, "transaction_type_POS": {"type": "integer", "maximum": 1, "minimum": 0}, "merchant_category_Food": {"type": "integer", "maximum": 1, "minimum": 0}, "transaction_type_Online": {"type": "integer", "maximum": 1, "minimum": 0}, "merchant_category_Travel": {"type": "integer", "maximum": 1, "minimum": 0}, "merchant_category_Grocery": {"type": "integer", "maximum": 1, "minimum": 0}, "merchant_category_Clothing": {"type": "integer", "maximum": 1, "minimum": 0}, "merchant_category_Electronics": {"type": "integer", "maximum": 1, "minimum": 0}}}

def get_max_range(feature):
    """Get the maximum value for a feature from the schema."""
    return schema["properties"].get(feature, {}).get("maximum", 1)

batch_size = 10
SCHEMA_ID = TRAINING_SCHEMA_ID
DATASET_SLUG = TRAINING_DATASET_SLUG

# Reuse agg_value if defined; otherwise define here
def agg_value(resp):
    recs = resp.get("records", [])
    if not recs:
        raise ValueError(f"Unexpected aggregation response: {resp}")
    rec0 = recs[0]
    if "data" in rec0 and isinstance(rec0["data"], dict) and "value" in rec0["data"]:
        v = rec0["data"].get("value")
        return float(v) if v is not None else 0.0
    if "value" in rec0:
        v = rec0.get("value")
        return float(v) if v is not None else 0.0
    raise ValueError(f"Unexpected aggregation response shape: {resp}")

# Get total count first
print("Getting total record count...")
count_resp = client.aggregate(
    organization=ORGANIZATION,
    dataset_slug=DATASET_SLUG,
    schema_slug=SCHEMA_SLUG,
    agg_filter="dataset_order:count(0~100000)",
    decrypt=False,
    schema_id=SCHEMA_ID
)
total_count = int(agg_value(count_resp))
print(f"Total records: {total_count}")
total_rows = total_count

# Compute aggregations over the entire dataset (not batched by transaction_id)
# Since transaction_ids are random/sparse, we aggregate over all records at once
print(f"\nComputing mean for each feature across all {total_count} records:\n")

for feature in features[:5]:
    try:
        # Use a wide range to cover all possible values
        max_range = get_max_range(feature)
        resp = client.aggregate(
            organization=ORGANIZATION,
            dataset_slug=DATASET_SLUG,
            schema_slug=SCHEMA_SLUG,
            agg_filter=f"{feature}:avg(0~{max_range})",
            decrypt=False,
            schema_id=SCHEMA_ID
        )
        mean_val = agg_value(resp)
        print(f"  mean {feature}: {mean_val:.3f}")
    except Exception as e:
        print(f"  Error with {feature}: {str(e)[:100]}")


print(f"\nStarting batch mean aggregation for each feature across using batch size {batch_size}:\n")

batch_data = []  # Collect batch results in a list
            
start = 0
while start < total_rows:
    end = min(start + batch_size - 1, total_rows - 1)
    batch_filter = f"dataset_order:{start}~{end}"
    print(f"\nBatch {start}-{end}:")

    data = {}
    for feature in features:
        max_range = get_max_range(feature)
        resp = client.aggregate(
            organization=ORGANIZATION,
            dataset_slug=DATASET_SLUG,
            schema_slug=SCHEMA_SLUG,
            agg_filter=f"{feature}:avg(0~{max_range})",
            extra_filters=[batch_filter],
            decrypt=False,
            schema_id=SCHEMA_ID
        )
        mean_val = agg_value(resp)
        data[feature] = mean_val
        print(f"  mean {feature}: {mean_val:.3f}")
    start += batch_size

    batch_data.append(data)  # Append to list instead

df = pd.DataFrame(batch_data)  # Create DataFrame from list of dicts
print(df)

print("\nAggregation complete!")

Getting total record count...
Total records: 500

Computing mean for each feature across all 500 records:

  mean amount: 916.804
  mean transaction_type_ATM: 0.250
  mean transaction_type_Online: 0.266
  mean transaction_type_POS: 0.234
  mean transaction_type_QR: 0.250

Starting batch mean aggregation for each feature across using batch size 10:


Batch 0-9:
  mean amount: 2511.100
  mean transaction_type_ATM: 0.400
  mean transaction_type_Online: 0.100
  mean transaction_type_POS: 0.200
  mean transaction_type_QR: 0.300
  mean merchant_category_Clothing: 0.100
  mean merchant_category_Electronics: 0.100
  mean merchant_category_Food: 0.300
  mean merchant_category_Grocery: 0.100
  mean merchant_category_Travel: 0.400
  mean country_DE: 0.200
  mean country_FR: 0.100
  mean country_NG: 0.200
  mean country_TR: 0.300
  mean country_UK: 0.100
  mean country_US: 0.100
  mean hour: 13.400
  mean is_fraud: 1.000

Batch 10-19:
  mean amount: 1251.200
  mean transaction_type_ATM: 0.600
  me

In [46]:
from sklearn.linear_model import LogisticRegression

df_features = df.iloc[:, :-1]
df_target = df.iloc[:, -1]

model = LogisticRegression()
model.fit(df_features, df_target)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`mul

In [41]:
model.score(df_features, df_target)

1.0

In [49]:
df_temp = df_target
df_temp[0] = 0.0
print(df_temp)

0     0.0
1     1.0
2     1.0
3     1.0
4     1.0
5     1.0
6     1.0
7     1.0
8     1.0
9     1.0
10    1.0
11    1.0
12    1.0
13    1.0
14    1.0
15    1.0
16    1.0
17    1.0
18    1.0
19    1.0
20    1.0
21    1.0
22    1.0
23    1.0
24    1.0
25    0.0
26    0.0
27    0.0
28    0.0
29    0.0
30    0.0
31    0.0
32    0.0
33    0.0
34    0.0
35    0.0
36    0.0
37    0.0
38    0.0
39    0.0
40    0.0
41    0.0
42    0.0
43    0.0
44    0.0
45    0.0
46    0.0
47    0.0
48    0.0
49    0.0
Name: is_fraud, dtype: float64


In [50]:
model.score(df_features, df_temp)

0.98